<a href="https://colab.research.google.com/github/ehurtos/MLOPS/blob/main/MLPOS_Streemlit_ML_model_for_Iris_Flower_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install streamlit

In [12]:
#Save the trained model as a pickle file
import pickle
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save the trained model as a pickle file
with open("iris_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as iris_model.pkl")


Model saved as iris_model.pkl


In [16]:
#Create the application to take the imput, calls the model and makes prediction
%%writefile app.py
import streamlit as st
import pickle
import numpy as np

st.title("Iris Flower Prediction")

st.write("Enter the following measurements to predict the Iris flower species:")

# Input fields for the four features
sepal_length = st.number_input("Sepal Length (cm)", min_value=0.0, step=0.1)
sepal_width = st.number_input("Sepal Width (cm)", min_value=0.0, step=0.1)
petal_length = st.number_input("Petal Length (cm)", min_value=0.0, step=0.1)
petal_width = st.number_input("Petal Width (cm)", min_value=0.0, step=0.1)

# Prediction function
def predict(input_features):
    try:  # Handle potential file errors
        with open("iris_model.pkl", "rb") as f:
            loaded_model = pickle.load(f)
        prediction = loaded_model.predict(input_features.reshape(1, -1))[0]  # Reshape for sklearn
        return prediction
    except FileNotFoundError:
        st.error("Error: iris_model.pkl not found. Make sure the model file is in the same directory as the app.")
        return None  # Or raise the exception if you want the app to stop
    except Exception as e:
        st.error(f"An error occurred during prediction: {e}")
        return None


if st.button("Predict"):  # Button to trigger prediction
    if sepal_length and sepal_width and petal_length and petal_width is not None:  # Check if all inputs are provided
        sample_input = np.array([sepal_length, sepal_width, petal_length, petal_width])

        prediction = predict(sample_input)

        if prediction is not None: # Check if the prediction was successful
            # Display the prediction result
            class_names = ['setosa', 'versicolor', 'virginica']  # Replace with your actual class names
            predicted_class = class_names[prediction]  # Assuming your model returns class indices
            st.write(f"Predicted Iris class: **{predicted_class}**")

    else:
        st.warning("Please enter all four measurements.")



Overwriting app.py


In [14]:
# to finf out which IP is the  Streemlit app is will be running on google cloud VM
import socket
import requests

# Get the hostname
hostname = socket.gethostname()

# Get the internal IP address
internal_ip = socket.gethostbyname(hostname)

# Get the external (public) IP address
try:
    external_ip = requests.get('https://api.ipify.org').text
except requests.exceptions.RequestException as e:
    external_ip = "Error: Could not retrieve external IP address"
    print(f"Error: {e}")

# Print the results
print(f"Hostname: {hostname}")
print(f"Internal IP: {internal_ip}")
print(f"External IP: {external_ip}")

Hostname: e08a5001d377
Internal IP: 172.28.0.12
External IP: 34.16.165.246


In [15]:
#Run the Streemlit Application for simple calulations . Note passord i the IP address
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.165.246:8501

⠼⠴⠦your url is: https://cold-bees-design.loca.lt
  Stopping...
^C
